In [10]:
# Packages
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests as req
from bs4 import BeautifulSoup as bs 
import json

In [12]:
# Youtube API Stuff **ASK PROF ABOUT THIS**

from googleapiclient.discovery import build 
from dotenv import load_dotenv
import os
load_dotenv()

api_key = os.getenv("YOUTUBE_API_KEY")
                    
youtube = build("youtube", "v3", developerKey=api_key)

In [13]:
#Pull video data check
video_ids = "Ks-_Mh1QhMc,VIDEO_ID_2"

request = youtube.videos().list(
    part="snippet,statistics",
    id=video_ids
)

response = request.execute()

for item in response["items"]:
    title = item["snippet"]["title"]
    views = item["statistics"].get("viewCount")
    likes = item["statistics"].get("likeCount")
    
    print(title, views, likes)

Your Body Language May Shape Who You Are | Amy Cuddy | TED 28173989 523935


In [24]:
# Put api data into a dataframe
youtube = []

In [25]:
df = pd.DataFrame(video_data, columns=["title", "views", "likes"])

In [26]:
df.head()

,title,views,likes


In [ ]:
# NOTE: 
# API Data is more up to date than Kaggle data, so view counts and likes are higher.
# Use kaggle data for details on videos, and API data for up to date stats on video performance. 
#**ASK Prof about API key and how to hide before I push to github 

In [42]:
# Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import kaggle
from googleapiclient.discovery import build
from dotenv import load_dotenv
import os

# Youtube API 
load_dotenv()
api_key = os.getenv("YOUTUBE_API_KEY")
youtube = build("youtube", "v3", developerKey=api_key)

# Kaggle Dataset
os.environ["KAGGLE_USERNAME"] = os.getenv("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = os.getenv("KAGGLE_KEY")
kaggle.api.authenticate()
kaggle.api.dataset_download_files("datasnaek/youtube-new", path="./data", unzip=True)
df_kaggle = pd.read_csv("./data/USvideos.csv")

print(df_kaggle.shape)
print(df_kaggle.head())


Dataset URL: https://www.kaggle.com/datasets/datasnaek/youtube-new
(40949, 16)
      video_id trending_date  \
0  2kyS6SvSYSE      17.14.11   
1  1ZAPwfrtAFY      17.14.11   
2  5qpjK5DgCt4      17.14.11   
3  puqaWrEC7tY      17.14.11   
4  d380meD0W0M      17.14.11   

                                               title          channel_title  \
0                 WE WANT TO TALK ABOUT OUR MARRIAGE           CaseyNeistat   
1  The Trump Presidency: Last Week Tonight with J...        LastWeekTonight   
2  Racist Superman | Rudy Mancuso, King Bach & Le...           Rudy Mancuso   
3                   Nickelback Lyrics: Real or Fake?  Good Mythical Morning   
4                           I Dare You: GOING BALD!?               nigahiga   

   category_id              publish_time  \
0           22  2017-11-13T17:13:01.000Z   
1           24  2017-11-13T07:30:00.000Z   
2           23  2017-11-12T19:05:24.000Z   
3           24  2017-11-13T11:00:04.000Z   
4           24  2017-11-12T18:01:

In [44]:
# Merge the YAPI and the KAPI 

print(df_kaggle["video_id"].head(10))
print(df_kaggle["video_id"].nunique())

print(df_kaggle["video_id"].nunique())


0    2kyS6SvSYSE
1    1ZAPwfrtAFY
2    5qpjK5DgCt4
3    puqaWrEC7tY
4    d380meD0W0M
5    gHZ1Qz0KiKM
6    39idVpFF7NQ
7    nc99ccSXST0
8    jr9QtXwC9vc
9    TUmyygCMMGA
Name: video_id, dtype: str
6351
6351


In [45]:
# Get unique video IDs
video_ids = df_kaggle["video_id"].dropna().unique().tolist()

# Fetch in batches of 50
rows = []
for i in range(0, len(video_ids), 50):
    batch = video_ids[i:i+50]
    request = youtube.videos().list(
        part="snippet,statistics",
        id=",".join(batch)
    )
    response = request.execute()
    
    for item in response["items"]:
        rows.append({
            "video_id":  item["id"],
            "title":     item["snippet"]["title"],
            "channel":   item["snippet"]["channelTitle"],
            "published": item["snippet"]["publishedAt"],
            "views":     int(item["statistics"].get("viewCount", 0)),
            "likes":     int(item["statistics"].get("likeCount", 0)),
            "comments":  int(item["statistics"].get("commentCount", 0)),
        })

df_api = pd.DataFrame(rows)
print(df_api.shape)
print(df_api.head())

(5688, 7)
      video_id                                              title  \
0  2kyS6SvSYSE                 WE WANT TO TALK ABOUT OUR MARRIAGE   
1  1ZAPwfrtAFY  The Trump Presidency: Last Week Tonight with J...   
2  puqaWrEC7tY                   Nickelback Lyrics: Real or Fake?   
3  d380meD0W0M                           I Dare You: GOING BALD!?   
4  gHZ1Qz0KiKM                              2 Weeks with iPhone X   

                 channel             published     views   likes  comments  
0           CaseyNeistat  2017-11-13T17:13:01Z   2983321  102173     20019  
1        LastWeekTonight  2017-11-13T07:30:00Z  23210890  262943     22449  
2  Good Mythical Morning  2017-11-13T11:00:04Z   1451223   21714      3199  
3               nigahiga  2017-11-12T18:01:41Z   5055903  190746     22325  
4               iJustine  2017-11-13T19:07:23Z   2190571   37697      3271  


In [ ]:
df = pd.merge(df_kaggle, df_api, on="video_id", how="inner")
print(df.shape)
print(df.columns.tolist())

# Little note: This is actually really interesting — 
# you can compare 2017 view/like counts (kaggle) vs current counts (API) to see how
# much videos have grown.

(37102, 22)
['video_id', 'trending_date', 'title_x', 'channel_title', 'category_id', 'publish_time', 'tags', 'views_x', 'likes_x', 'dislikes', 'comment_count', 'thumbnail_link', 'comments_disabled', 'ratings_disabled', 'video_error_or_removed', 'description', 'title_y', 'channel', 'published', 'views_y', 'likes_y', 'comments']


In [48]:
# Little cleaning

df = df.drop(columns=["title_y", "channel"])
df = df.rename(columns={
    "title_x":        "title",
    "views_x":        "views_2017",
    "likes_x":        "likes_2017",
    "views_y":        "views_current",
    "likes_y":        "likes_current",
    "comment_count":  "comments_2017",
    "comments":       "comments_current"
})

print(df.columns.tolist())

['video_id', 'trending_date', 'title', 'channel_title', 'category_id', 'publish_time', 'tags', 'views_2017', 'likes_2017', 'dislikes', 'comments_2017', 'thumbnail_link', 'comments_disabled', 'ratings_disabled', 'video_error_or_removed', 'description', 'published', 'views_current', 'likes_current', 'comments_current']


In [ ]:
# Null and type check
print(df.dtypes)
print(df.isnull().sum())

video_id                    str
trending_date               str
title                       str
channel_title               str
category_id               int64
publish_time                str
tags                        str
views_2017                int64
likes_2017                int64
dislikes                  int64
comments_2017             int64
thumbnail_link              str
comments_disabled          bool
ratings_disabled           bool
video_error_or_removed     bool
description                 str
published                   str
views_current             int64
likes_current             int64
comments_current          int64
dtype: object
video_id                    0
trending_date               0
title                       0
channel_title               0
category_id                 0
publish_time                0
tags                        0
views_2017                  0
likes_2017                  0
dislikes                    0
comments_2017               0
thumbnail_link  

In [50]:
# Cleaning: Convert formats of dates to datetime objects
df["trending_date"] = pd.to_datetime(df["trending_date"], format="%y.%d.%m")
df["publish_time"] = pd.to_datetime(df["publish_time"])
df["published"] = pd.to_datetime(df["published"])

print(df[["trending_date", "publish_time", "published"]].dtypes)
print(df[["trending_date", "publish_time", "published"]].head())

trending_date         datetime64[us]
publish_time     datetime64[us, UTC]
published        datetime64[us, UTC]
dtype: object
  trending_date              publish_time                 published
0    2017-11-14 2017-11-13 17:13:01+00:00 2017-11-13 17:13:01+00:00
1    2017-11-14 2017-11-13 07:30:00+00:00 2017-11-13 07:30:00+00:00
2    2017-11-14 2017-11-13 11:00:04+00:00 2017-11-13 11:00:04+00:00
3    2017-11-14 2017-11-12 18:01:41+00:00 2017-11-12 18:01:41+00:00
4    2017-11-14 2017-11-13 19:07:23+00:00 2017-11-13 19:07:23+00:00
